# Data Consumption — Cymatics Classification (Pattern image Search)

Three search modes for finding similar cymatics patterns:
1. **Upload an image** → CLIP image embedding → Milvus ANN search
2. **Record audio** → generate cymatics image → CLIP → Milvus
3. **Text query** → CLIP text embedding → Milvus (text-to-image)

Prerequisites:
- Milvus running (`docker compose up -d milvus`)
- Cymatics embeddings ingested (orchestrator → option 6)

## Environment setup

In [ ]:
from pathlib import Path
import os
import sys

_here = Path.cwd().resolve()
_root = next(
    (p for p in [_here, *_here.parents]
     if (p / "docker-compose.yml").is_file() and (p / "orchestrate.py").is_file()),
    None,
)
if _root is None:
    raise RuntimeError("Repo root not found — open the notebook from the BDM-Cymatics tree")

PROJECT_ROOT = str(_root)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from dotenv import load_dotenv
load_dotenv(_root / ".env", override=False)

print("PROJECT_ROOT =", PROJECT_ROOT)

## Constants

In [ ]:
import numpy as np
import cv2

SAMPLE_RATE = 44100
DURATION = 5
TOP_K = 5
IMG_SIM = 900    # Simulation grid resolution
IMG_RES = 2048   # Final cymatics image resolution

## Connect to Milvus

In [ ]:
from pymilvus import MilvusClient

MILVUS_URI = os.environ.get("MILVUS_URI", "http://localhost:19530")
milvus_client = MilvusClient(uri=MILVUS_URI)
print(f"Milvus connected: {MILVUS_URI}")
print(f"Collections: {milvus_client.list_collections()}")

## Audio recording — capture 5 s mono from the default microphone

In [ ]:
import sounddevice as sd

def record_audio(duration=DURATION, sample_rate=SAMPLE_RATE):
    # Record mono audio, return float32 array normalised to [-1, 1].
    print(f"\n  Recording {duration}s of audio...")
    audio = sd.rec(int(duration * sample_rate), samplerate=sample_rate,
                   channels=1, dtype="float32")
    sd.wait()
    audio = audio.flatten()
    peak = np.abs(audio).max()
    if peak > 0:
        audio = audio / peak
    print(f"  Recording complete — {len(audio)} samples")
    return audio

## Cymatics image generation — replicate trusted-zone pipeline to produce a PNG

In [ ]:
from shared.cymatics_engine import (
    N_ZONES, build_zones, build_zone_sources, analyse_frame,
    compute_interference, displacement_to_brightness, render_composite,
)
from shared.freq_detection import build_window_candidates, harmonic_dominant_freq

def generate_cymatics_image(audio, sample_rate=SAMPLE_RATE):
    # Generate a cymatics PNG from audio. Returns raw PNG bytes.
    # Step 1: Normalise to float64 [-1, 1].
    audio = audio.astype(np.float64)
    pk = np.max(np.abs(audio))
    if pk > 1e-6:
        audio /= pk

    # Step 2: Detect dominant frequency and find best 0.25 s chunk.
    candidates = build_window_candidates(audio, sample_rate)
    dom_freq = harmonic_dominant_freq(candidates, max_harmonics=4, use_energy_weight=True)
    dom_bin = int(round(dom_freq))

    if dom_bin > 0:
        matching = [c for c in candidates if c[0] > 0 and c[0] % dom_bin == 0]
        best = max(matching, key=lambda c: c[3]) if matching else candidates[0]
    else:
        best = max(candidates, key=lambda c: c[3]) if candidates else candidates[0]

    peak_chunk = np.asarray(best[1], dtype=np.float64)
    peak_rms = float(np.sqrt(np.mean(peak_chunk ** 2)))
    peak_time = best[2] / sample_rate
    print(f"  Detected peak frequency: {best[0]} Hz")

    # Step 3: Build simulation geometry and compute interference patterns.
    iz = build_zones(IMG_SIM)
    i_sources = build_zone_sources(IMG_SIM, iz)
    img_zone_br = []
    for zi in range(N_ZONES):
        _, sfreqs, _ = analyse_frame(peak_chunk, zi, sample_rate)
        disp_x, disp_y = compute_interference(i_sources[zi], sfreqs, peak_time, IMG_SIM)
        br = displacement_to_brightness(disp_x, disp_y, iz["masks"][zi], IMG_SIM, peak_rms, 0.0)
        img_zone_br.append(br)

    # Step 4: Render composite image and upscale to 2048x2048.
    raw_img = render_composite(img_zone_br, iz, 0.0, IMG_SIM)
    img = cv2.resize(raw_img, (IMG_RES, IMG_RES), interpolation=cv2.INTER_LANCZOS4)

    # Step 5: Glow post-processing (matches trusted zone).
    glow = cv2.GaussianBlur(img, (0, 0), sigmaX=14)
    img = cv2.addWeighted(img, 0.82, glow, 0.30, 0)

    # Step 6: Encode to PNG bytes for CLIP embedding.
    success, buf = cv2.imencode(".png", img)
    if not success:
        raise RuntimeError("Failed to encode cymatics image to PNG.")
    return bytes(buf)

## CLIP embedding helpers — image and text

In [ ]:
import torch
from PIL import Image
from io import BytesIO

try:
    import clip
    clip_model, clip_preprocess = clip.load("ViT-B/32", device="cpu")
    print("CLIP ViT-B/32 loaded")
except Exception as e:
    print(f"CLIP not available: {e}")
    clip_model = None

def clip_image_embedding(image_bytes):
    # Compute CLIP image embedding (512-dim) from PNG bytes.
    img = Image.open(BytesIO(image_bytes)).convert("RGB")
    img_tensor = clip_preprocess(img).unsqueeze(0)
    with torch.no_grad():
        emb = clip_model.encode_image(img_tensor).squeeze(0).float().numpy()
    emb = emb / (np.linalg.norm(emb) + 1e-12)
    return emb

def clip_text_embedding(text):
    # Compute CLIP text embedding (512-dim) from a string.
    tokens = clip.tokenize([text])
    with torch.no_grad():
        emb = clip_model.encode_text(tokens).squeeze(0).float().numpy()
    emb = emb / (np.linalg.norm(emb) + 1e-12)
    return emb

## Search Milvus for similar cymatics patterns

In [ ]:
CYMATICS_COLLECTION = "sound_cymatics_embeddings"

def search_patterns(embedding, top_k=TOP_K):
    # Search the Milvus cymatics embedding collection.
    results = milvus_client.search(
        collection_name=CYMATICS_COLLECTION,
        data=[embedding.tolist()],
        limit=top_k,
        output_fields=["uuid", "category", "source", "peak_frequency_hz",
                       "symmetry_score", "image_path"],
        search_params={"metric_type": "COSINE", "params": {"nprobe": 16}},
    )
    return results[0] if results else []

def display_pattern_results(results, title="Pattern Search Results"):
    print(f"\n{'=' * 62}")
    print(f"  {title} — Top {len(results)}")
    print(f"{'─' * 62}")
    if not results:
        print("  No matching patterns found.")
        return
    for i, hit in enumerate(results):
        e = hit["entity"]
        print(f"  {i+1}. Similarity: {hit['distance']:.4f}")
        print(f"     UUID:       {e.get('uuid', '?')}")
        print(f"     Category:   {e.get('category', '') or '—'}")
        print(f"     Peak freq:  {e.get('peak_frequency_hz', 0):.0f} Hz")
        print(f"     Image:      {e.get('image_path', '') or '—'}")
        print()
    print(f"{'=' * 62}\n")

## Mode 1: Search by uploaded image

In [ ]:
IMAGE_PATH = "/path/to/your/cymatics/image.png"
with open(IMAGE_PATH, "rb") as f:
    image_bytes = f.read()
emb = clip_image_embedding(image_bytes)
results = search_patterns(emb)
display_pattern_results(results, "Image Pattern Search Results")


## Mode 2: Record audio → generate cymatics → search

In [ ]:
audio = record_audio()
image_bytes = generate_cymatics_image(audio)
emb = clip_image_embedding(image_bytes)
results = search_patterns(emb)
display_pattern_results(results, "Audio → Cymatics Pattern Search Results")


## Mode 3: Search by text query

In [ ]:
QUERY = "symmetric circular pattern with high frequency"
emb = clip_text_embedding(QUERY)
results = search_patterns(emb)
display_pattern_results(results, f"Text → Pattern Results: '{QUERY}'")
